In [20]:
import os
import openai
import random
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")


In [21]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.schema.output_parser import StrOutputParser
from IPython.display import Image, display
from typing import Literal



import json

In [22]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [31]:
class Agent:
     def __init__(self, model ):
     
          self.model=model
          graph = StateGraph(AgentState)

          graph.add_node("Entrada", self.entrada)
          graph.add_node("Maçã", self.Maçã)
          graph.add_node("Banana", self.Banana)

          graph.set_entry_point("Entrada")
          graph.add_conditional_edges("Entrada", self.sortear)
          graph.add_edge("Banana", END)
          graph.add_edge("Maçã", END)
          self.graph = graph.compile()

     def entrada(self, state: AgentState):
          return state


     def sortear(self, state: AgentState):
         
          if random.random() < 0.5:
               return "Maçã"
          else: 
               return "Banana"
          
     def Maçã(self, state: AgentState):
          novo_state = {
               'messages' : state ['messages'] + [SystemMessage(content="Maçã")]
          }
          return self.resposta(novo_state)
     
     def Banana(self, state: AgentState):
          novo_state = {
               'messages' : state ['messages'] + [SystemMessage(content="Banana")]
          }
          return self.resposta(novo_state)
         

     def resposta(self, state: AgentState):     
          prompt = """Com base na fruta retornada, explique qual os principais benefícios da ingestão dela."""
          messages = state['messages']
          resposta = self.model.invoke([HumanMessage(content=prompt + " - " + state["messages"][-1].content)])

          mensagem_final = state["messages"][-1].content + " - " + resposta.content
          
          return {'messages': [mensagem_final]}
         

In [32]:


model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
chain = Agent(model)




In [33]:
entrada = {"messages": [HumanMessage]}
result = chain.graph.invoke(entrada)

In [34]:
result['messages'][-1]

'Banana - A banana é uma fruta muito popular e nutritiva, conhecida por seus diversos benefícios à saúde. Aqui estão os principais benefícios da ingestão de banana:\n\n1. **Fonte de Energia Rápida**: A banana é rica em carboidratos, principalmente na forma de açúcares naturais (glicose, frutose e sacarose), que fornecem energia rápida e sustentável, ideal para atletas e pessoas ativas.\n\n2. **Rica em Potássio**: Uma das características mais conhecidas da banana é seu alto teor de potássio, um mineral essencial para o funcionamento adequado do coração, músculos e sistema nervoso. O potássio ajuda a regular a pressão arterial e a prevenir cãibras musculares.\n\n3. **Melhora a Digestão**: A banana contém fibras, especialmente a pectina, que ajuda a regular o trânsito intestinal, prevenindo a constipação e promovendo a saúde digestiva.\n\n4. **Fonte de Vitaminas e Minerais**: Além do potássio, a banana fornece vitamina C, vitamina B6, magnésio e outros nutrientes importantes para o sistem